# Kalibratie — trainingen scoren op herschrijfbaarheid

Draai de scorer op een handvol trainingen, bekijk de output, en leg naast je handmatige labels.

**Voor je begint:**
1. `pip install -r requirements.txt`
2. Zet je API-key in een `.env`-bestand (zie `.env.example`).
3. Download de Google Sheet als **xlsx** (Bestand → Downloaden) en zet het pad hieronder.
4. Zet `Scoringsrubric_herschrijfbaarheid_v1.md` in dezelfde map.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import importlib, score_trainings as st
importlib.reload(st)   # zodat je edits in score_trainings.py meteen meepakt

IN_XLSX   = "trainingen.xlsx"                          # <- jouw gedownloade sheet
RUBRIC    = "Scoringsrubric_herschrijfbaarheid_v1.md"
N         = 10                                         # eerst een klein setje
WEB_SEARCH = True                                      # actualiteitscheck aan

## 1. Sheet inlezen en de content-parser controleren
Check eerst dat de kolommen goed herkend worden en dat de brontekst schoon uit de JSON komt.

In [ ]:
df, cols = st.read_input(IN_XLSX)
print("Herkende kolommen:", cols)
print("Aantal rijen:", len(df))

# Inspecteer de eerste training: parsed content + geschoonde brontekst
row0 = df.iloc[0]
content0 = st.parse_content(row0[cols['content']])
print("\nJSON-keys:", list(content0))
print("Dagen:", st.extract_days(content0, row0[cols['days']] if cols['days'] else None))
print("\n--- brontekst (eerste 800 tekens) ---")
print(st.build_source_text(content0, str(row0[cols['name']]))[:800])

## 2. Scoren
Kost een paar seconden per training (meer met web search). Output gaat ook naar een xlsx.

In [ ]:
out = st.run_file(IN_XLSX, "trainingen_scored.xlsx", RUBRIC, limit=N, use_web_search=WEB_SEARCH)
out[['titel','eindscore','verdict','actualiteit_type','actualiteit_severity','menselijke_input_nodig','scorer_confidence']]

## 3. Naast je handmatige labels leggen
Vul je eigen labels in (id → score). Kijk of de scores correleren en of de verdict-grenzen landen waar je ze wil.
Stel daarna in `score_trainings.py` de `IMPACT`-tabel of `WEIGHTS` bij en draai opnieuw (reload-cel bovenaan).

In [ ]:
import pandas as pd

# jouw handmatige labels: {training_id: score}
labels = {
    # 43: 40,   # Cursus XSL
    # 87: 70,   # Cursus LDAP
}

cmp = out[['training_id','titel','eindscore','verdict']].copy()
cmp['label'] = cmp['training_id'].map(labels)
cmp['delta'] = (cmp['eindscore'] - cmp['label']).abs()
cmp.dropna(subset=['label'])

In [ ]:
# Verdeling van de scores over de populatie — zie hoe scheef je input is
out['verdict'].value_counts().reindex(['al_nieuwe_stijl','rijk','redelijk','dun','onbruikbaar']).fillna(0).astype(int)

## 4. Volledige feedback van één training bekijken
Handig om te zien of de vooruitgerichte feedback (bruikbaar / strippen / gaten / rewrite_guidance) klopt.

In [ ]:
import textwrap
r = out.iloc[0]
for k in ['titel','kern','eindscore','verdict','actualiteit_actie','bruikbaar','strippen','gaten','rewrite_guidance']:
    print(f"{k}:")
    print(textwrap.fill(str(r[k]), 100, initial_indent='  ', subsequent_indent='  '))
    print()